[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.05.notebook-title|Introduction -->
# Week 5: Advanced pandas
<!-- /gen -->

*This week you reshape data, read trends with window functions, and summarise several statistics at once.*

## This week at Runnymede

Week 4 covered loading, filtering, joining, and aggregating `DataFrames`. This week the client questions are **harder** (oops). While the answer is quick to compute, this is the case only once the data is in the right **shape**.

A director who asks "are weekend promotions working?" does not want thousands of rows of store-level sales. They want one small table with regions down the side and promo and no-promo across the top. Turning the first into the second is called **reshaping**. It is the core skill of Week 5 (and beyond).

This week's [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) client is **Marlow & Finch**, a home-and-gifts retail chain. The deliverable is one table for a board pack.

You work in your GitHub Codespace, with GitHub Copilot as your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate). [SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) records your Copilot chats. If anything does not work, the [Your workstation guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/codespaces-start.md) explains how to fix it.

## This week's learning objectives

The [week README](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/05/README.md) lists this week's learning objectives.

## pandas, continued

Week 5 is the second of two pandas weeks, followed by Week 6 where we start looking at data visualisation design. In Week 7, your team runs its [Sprint Zero](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-sprint-zero) on the [London Fire Brigade data](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/lfb-data.md), at its own pace. The [module README](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/README.md) lists each week's topic.

## Key concept 1: Tidy data

"Tidy data" ([Hadley Wickham](https://en.wikipedia.org/wiki/Hadley_Wickham)'s term) is a standard way to lay out a table for analysis tools:

1. Each **variable** forms a **column**
2. Each **observation** forms a **row**
3. Each **type of observational unit** forms a **table**

Run the cell below. It prints the same sales data twice, first in *wide* form and then in *long* (tidy) form. You convert with the [`melt()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html) function.

In [ ]:
# --- Data setup: Small in-memory examples for this introduction ---
import numpy as np
import pandas as pd

rng = np.random.default_rng(5813)  # module seed: Do not change

# Wide: Months as columns. Lovely in a report, awkward for analysis
wide = pd.DataFrame({
    "store": ["Marlow", "Henley"],
    "Jan": [41200, 33100],
    "Feb": [39800, 35400],
    "Mar": [44100, 36900],
})
print("Wide format:")
print(wide)

# Long (tidy): One row per store per month
long = wide.melt(id_vars="store", var_name="month", value_name="sales")
print("\nLong (tidy) format:")
print(long)

Neither format is "correct". They serve different **readers**:

- **Long** is for analysis: Grouping, filtering, merging, and charting tools all expect one observation per row
- **Wide** is for people: A board table has categories across the top, because a reader compares values across a row

Go back to the above and see for yourself.

The usual workflow starts with wide data. You `melt` it to long, analyse it, then pivot back to a small wide table for the audience. You do this loop in the Demonstration and again in the quest.

## Key concept 2: The reshaping verbs

| Verb | Direction | When |
|------|-----------|------|
| `melt()` | wide to long | Column headers that are really values (months, years, regions) |
| `pivot_table()` | long to wide | A summary grid, with aggregation when several rows land in one cell |
| `unstack()` | index to columns | A two-level row label from grouping, with one level wanted across the top |
| `stack()` | columns to index | The reverse of `unstack()` |

Run the cell below. One line of [`pivot_table()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) turns a long table of daily sales into a briefing-ready grid.

In [ ]:
# A taste of pivot_table(): 90 days of sales for two stores and two channels
days = pd.date_range("2026-01-01", periods=90, freq="D")
taste = pd.DataFrame({
    "date": np.tile(days, 4),
    "store": np.repeat(["Marlow", "Marlow", "Henley", "Henley"], 90),
    "channel": np.repeat(["in-store", "online"] * 2, 90),
    "sales": rng.normal(1000, 150, 360).round(2),
})

summary = taste.pivot_table(values="sales", index="store",
                            columns="channel", aggfunc="mean").round(0)
print(summary)

**Optional:** A quick note on how functions (and their list of parameters) are formatted. Compare the following:

In [ ]:
summary_a = taste.pivot_table(values="sales", index="store",
                              columns="channel", aggfunc="mean").round(0)

summary_b = taste.pivot_table(
    values="sales",
    index="store",
    columns="channel",
    aggfunc="mean"
    ).round(0)

The standard way of formatting Python code is `a` above. Personally, I find `b` easier to read and to use. For example, when I want to test the same assignment without, say, `aggfunc`, all I have to do is put a `#` before that row. This converts it into a comment, which Python ignores. Given that Jupyter notebooks **invite** experimentation, I find that much more useful (as, I'm sure, you will come to appreciate once we come to data visualisation).

In [ ]:
summary_c = taste.pivot_table(
    values="sales",
    index="store",
    columns="channel",  # You should also remove the trailing `,` although it'll work without
    # aggfunc="mean"
    ).round(0)

## Key concept 3: `MultiIndex`

Group by two keys and pandas gives the result a [**`MultiIndex`**](https://pandas.pydata.org/docs/user_guide/advanced.html), a row label with two levels (store, then channel). Here you read it, slice it with `.loc`, and [`unstack()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.unstack.html) a level into columns. Run the cell below to see a two-level result, then the same figures with the inner level unstacked into columns.

In [ ]:
by_store_channel = taste.groupby(
    ["store", "channel"]
    )["sales"].mean().round(0)

print("MultiIndex Series (two index levels):")
print(by_store_channel)

print("\nUnstacked (inner level becomes columns):")
print(by_store_channel.unstack())

## Key concept 4: Window functions

Daily sales are notoriously noisy. For example, weather, paydays, and one large tour group attending an event all influence a single day's total. [Window functions](https://pandas.pydata.org/docs/user_guide/window.html) compute each row's value from a window of neighbouring rows, which **separates trend from noise**:

- `rolling(7).mean()`: The 7-day moving average, the "smoothed" line
- `shift(7)`: The value from 7 rows earlier, the basis of week-on-week comparisons
- `pct_change()`: Percentage change from the previous row

Each of them assumes the rows are **sorted**, usually by date. *(Consequently, the most common window-function bug is running them on unsorted data.)*

Run the cell below to see a 7-day average and a week-on-week change for one store and one channel.

In [ ]:
one_store = (taste[(taste["store"] == "Marlow") & (taste["channel"] == "in-store")]
             .sort_values("date")
             .set_index("date"))

trend = pd.DataFrame({
    "sales": one_store["sales"],
    "rolling_7": one_store["sales"].rolling(7).mean().round(1),
    "vs_last_week_%": (one_store["sales"].pct_change(7) * 100).round(1),
})
print(trend.head(10))

## Key concept 5: Complex `groupby()`

Week 4's `groupby()` computes one statistic per group. Real <!-- plain-language: everyday brief -->briefs want several at once, with a different statistic for each column. That is `agg()` with a dictionary, or the tidier [*named aggregation*](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation). Sometimes you need a group statistic copied back onto every row, such as each row's share of its group total. That is [`transform()`](https://pandas.pydata.org/docs/user_guide/groupby.html#transformation). Run the cell below to see named aggregation and `transform()`.

In [ ]:
# Named aggregation: Several statistics in one pass, with readable column names
channel_summary = taste.groupby("channel").agg(
    total=("sales", "sum"),
    typical_day=("sales", "mean"),
    best_day=("sales", "max"),
).round(0)
print(channel_summary)

# transform(): Each row's share of its store's total
taste["share_of_store_%"] = (taste["sales"]
                             / taste.groupby("store")["sales"].transform("sum")
                             * 100).round(2)
print("\nSame number of rows as before, with a new per-row column:")
print(taste.head(3))

## Shape descriptions for your Associate

Reshaping is a really powerful example of demonstrating how your AI Associate can help. It's where **describing** what you want really beats **writing** code yourself.

For example, this is probably all your Associate needs:

> *"Rows = regions, columns = promo vs no-promo, cells = average daily sales, rounded to whole pounds"*

Give your Associate that sentence and it produces the code. Give it "analyse the promo data" and you get a plausible table that answers a question ... unfortunately, it may be a question nobody had asked. Hence, the skill this week is writing the shape sentence. Then **verify one cell of the result by hand** before you trust the grid.

Watch for these failure modes:

- `pivot_table` has a silent default, `aggfunc="mean"`: It averages when your business logic needs a sum
- `unstack()` fills missing combinations with `NaN`: An Associate may report those as zero

The quest's [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) stage uses both.

## Prerequisites

You need Week 4's fundamentals: Creating and filtering `DataFrames`, `read_csv()`, merging, basic `groupby()`, and handling missing values. If you are unsure of any of them, skim [Week 4's Demonstration](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/04/Demonstration.ipynb) first.

## Next steps

The [week README](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/05/README.md) lists what to do before class, what happens in class, and what to finish at home. The [AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens) callouts in the [Demonstration](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/05/Demonstration.ipynb) are prompts to try with your Associate as you go.